# NpuKit — int8 matmul on PYNQ-Z2

Loads `npukit.bit` (+ `.hwh` for DMA) and runs visible cases:

1. **Classic 8x8** — prints **A**, **B**, **C_npu**, **C_ref** for each case
2. **Tiled** — 16x16 demo + larger MxKxN, with a **tiling plan** (what K means)

Math write-up: `docs/tiling.md` in the repo.

Keep `npukit.bit`, `npukit.hwh`, and `npukit_matmul.py` beside this notebook.
After **Run All**, save the notebook so the matrix dumps stay in the file.

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_matmul as nk

importlib.reload(nk)
nk.VERBOSE = True  # print A/B/C + tiling for every case

mmio, transport = nk.open_device(BIT)
ident = mmio.read(nk.REG_ID)
assert ident == nk.ID_MAGIC, f"BAD ID 0x{ident:08X}"
print(
    f"ID OK version=0x{mmio.read(nk.REG_VERSION):08X} "
    f"N={mmio.read(nk.REG_N)} transport={type(transport).__name__}"
)

Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID OK version=0x00000300 N=8 transport=DmaTransport


## What tiling means (no FPGA run)

For `C = A @ B` with shapes `A[M x K]`, `B[K x N]`:

- **K** is the inner dimension (dot-product length).
- Hardware only multiplies **8x8** tiles; the host walks spatial blocks of C and, for each block, sums partial products over chunks of K.

In [2]:
print("Example plan for 16x16x16:")
print(nk.describe_tiling(16, 16, 16))
print()
print("Example plan for 32x32x32:")
print(nk.describe_tiling(32, 32, 32))

Example plan for 16x16x16:
  Shapes: A[16×16] @ B[16×16] → C[16×16]  (hardware tile T=8)
  K is the inner dim: each C[i,j] = sum_{p=0..15} A[i,p]*B[p,j]
  Spatial C tiles: 2×2 = 4   |   K-steps per C tile: 2
  Hardware runs: 2×2×2 = 8
  Example for C[0:T, 0:T]:
    k0=  0  CTRL=CLEAR|START  A[0:8, 0:8] @ B[0:8, 0:8]
    k0=  8  CTRL=START       A[0:8, 8:16] @ B[8:16, 0:8]
  Then repeat for every (i0,j0) in {0,8,…,8} × {0,8,…,8}.

Example plan for 32x32x32:
  Shapes: A[32×32] @ B[32×32] → C[32×32]  (hardware tile T=8)
  K is the inner dim: each C[i,j] = sum_{p=0..31} A[i,p]*B[p,j]
  Spatial C tiles: 4×4 = 16   |   K-steps per C tile: 4
  Hardware runs: 4×4×4 = 64
  Example for C[0:T, 0:T]:
    k0=  0  CTRL=CLEAR|START  A[0:8, 0:8] @ B[0:8, 0:8]
    k0=  8  CTRL=START       A[0:8, 8:16] @ B[8:16, 0:8]
    k0= 16  CTRL=START       A[0:8, 16:24] @ B[16:24, 0:8]
    k0= 24  CTRL=START       A[0:8, 24:32] @ B[24:32, 0:8]
  Then repeat for every (i0,j0) in {0,8,…,24} × {0,8,…,24}.


## Classic 8x8 suite

Each case prints A, B, FPGA result `C_npu`, and NumPy `C_ref`.

In [3]:
import numpy as np

rng = np.random.default_rng(0)
classic = nk.classic_8x8_cases(rng)
p0, t0 = nk.run_suite(mmio, transport, classic, verbose=True)
print(f"\nclassic: {p0}/{t0} PASS")
assert p0 == t0

CASE: demo rows×ones  [PASS]
  timing: npu=   8.651 ms   cpu(NumPy)=   0.199 ms   transport=DmaTransport
  Shapes: A[8×8] @ B[8×8] → C[8×8]  (hardware tile T=8)
  K is the inner dim: each C[i,j] = sum_{p=0..7} A[i,p]*B[p,j]
  Spatial C tiles: 1×1 = 1   |   K-steps per C tile: 1
  Hardware runs: 1×1×1 = 1
  Example for C[0:T, 0:T]:
    k0=  0  CTRL=CLEAR|START  A[0:8, 0:8] @ B[0:8, 0:8]

A shape=(8, 8) dtype=int8
[[   1    1    1    1    1    1    1    1]
 [   2    2    2    2    2    2    2    2]
 [   3    3    3    3    3    3    3    3]
 [   4    4    4    4    4    4    4    4]
 [   5    5    5    5    5    5    5    5]
 [   6    6    6    6    6    6    6    6]
 [   7    7    7    7    7    7    7    7]
 [   8    8    8    8    8    8    8    8]]

B shape=(8, 8) dtype=int8
[[   1    1    1    1    1    1    1    1]
 [   1    1    1    1    1    1    1    1]
 [   1    1    1    1    1    1    1    1]
 [   1    1    1    1    1    1    1    1]
 [   1    1    1    1    1    1    1    

## Tiled suite

Includes a readable **16x16** demo (A = row constants, B = I) plus a random **32x32x32**.

In [4]:
M, K, Ndim = 32, 32, 32  # must be multiples of 8
tiled = nk.tiled_cases(M, K, Ndim, rng)
p1, t1 = nk.run_suite(mmio, transport, tiled, verbose=True)
print(f"\ntiled: {p1}/{t1} PASS")
assert p1 == t1
print(f"\nall: {p0 + p1}/{t0 + t1} PASS")

CASE: tiled demo 16×16 (A rows× I)  [PASS]
  timing: npu=  33.319 ms   cpu(NumPy)=   0.204 ms   transport=DmaTransport
  Shapes: A[16×16] @ B[16×16] → C[16×16]  (hardware tile T=8)
  K is the inner dim: each C[i,j] = sum_{p=0..15} A[i,p]*B[p,j]
  Spatial C tiles: 2×2 = 4   |   K-steps per C tile: 2
  Hardware runs: 2×2×2 = 8
  Example for C[0:T, 0:T]:
    k0=  0  CTRL=CLEAR|START  A[0:8, 0:8] @ B[0:8, 0:8]
    k0=  8  CTRL=START       A[0:8, 8:16] @ B[8:16, 0:8]
  Then repeat for every (i0,j0) in {0,8,…,8} × {0,8,…,8}.

A shape=(16, 16) dtype=int8
[[   1    1    1    1    1    1    1    1    1    1    1    1    1    1    1    1]
 [   2    2    2    2    2    2    2    2    2    2    2    2    2    2    2    2]
 [   3    3    3    3    3    3    3    3    3    3    3    3    3    3    3    3]
 [   4    4    4    4    4    4    4    4    4    4    4    4    4    4    4    4]
 [   5    5    5    5    5    5    5    5    5    5    5    5    5    5    5    5]
 [   6    6    6    6    6    6

## Optional: full CLI re-run

Reloads the bitstream and prints the same verbose suite.

In [5]:
%run /home/xilinx/jupyter_notebooks/npukit_matmul.py /home/xilinx/jupyter_notebooks/npukit.bit 32 32 32

Using AXI DMA transport (/home/xilinx/jupyter_notebooks/npukit.bit)
ID OK version=0x00000300 N=8
NPU time includes DMA/MMIO + poll; CPU is NumPy matmul.
Verbose matrix dump on (use --quiet to suppress).

--- classic 8×8 ---
CASE: demo rows×ones  [PASS]
  timing: npu=   8.433 ms   cpu(NumPy)=   0.127 ms   transport=DmaTransport
  Shapes: A[8×8] @ B[8×8] → C[8×8]  (hardware tile T=8)
  K is the inner dim: each C[i,j] = sum_{p=0..7} A[i,p]*B[p,j]
  Spatial C tiles: 1×1 = 1   |   K-steps per C tile: 1
  Hardware runs: 1×1×1 = 1
  Example for C[0:T, 0:T]:
    k0=  0  CTRL=CLEAR|START  A[0:8, 0:8] @ B[0:8, 0:8]

A shape=(8, 8) dtype=int8
[[   1    1    1    1    1    1    1    1]
 [   2    2    2    2    2    2    2    2]
 [   3    3    3    3    3    3    3    3]
 [   4    4    4    4    4    4    4    4]
 [   5    5    5    5    5    5    5    5]
 [   6    6    6    6    6    6    6    6]
 [   7    7    7    7    7    7    7    7]
 [   8    8    8    8    8    8    8    8]]

B shape=(8, 8)